# Tool atrophy pilot: does tool-RL erode the skill the tool replaces?

One model family, three checkpoints from the same base, evaluated **with the tool
turned off**: the base model, a tool-RL checkpoint (trained with a Python
interpreter), and a text-only RL checkpoint (same base, same kind of data, no
tool). Same prompt, same decoding, pass@k plus an arithmetic error rate.

| Readout | Meaning |
|---|---|
| tool-RL below text-RL at high k, or more arithmetic slips | **atrophy** |
| tool-RL above both at every k | **internalization** |
| tool-RL and text-RL overlap at every k | tool added nothing beyond RL, stop |

The only published hint is CIR (2505.24480) Fig. 6: one 7B model, no text-RL
control. This notebook adds the control and a direct skill metric.

**What runs.** `src/gen.py` samples one checkpoint under each condition with vLLM
and writes one JSONL per (model, condition) to Google Drive, checkpointing every
`CHUNK` problems, so a disconnect costs at most one chunk. `src/analyze.py` turns the JSONL into pass@k curves,
bootstrap CIs, and a verdict. Every model gets the same grid, so each condition is
read as its own three-way comparison.

| Condition | Prompt | If the model writes a code block |
|---|---|---|
| `prompt_off` | shared no-code prompt ("do not write code") | intercepted, denial message injected, nothing runs |
| `env_denied` | the model's tool prompt (asks for code) | intercepted, denial message injected, nothing runs |
| `tool_on` | the model's tool prompt | executed with `python -I`, 4-call budget, then the budget message |
| `plain_cot` (optional) | shared plain CoT prompt | nothing intercepted; the model fabricates its own output block. CIR's condition. |

**Hardware.** Pick a GPU runtime. A100 40 GB runs everything. L4 24 GB runs the
1.5B preset comfortably and the 7B preset slowly. T4 only fits 1.5B (fp16).
Rough wall-clock on an A100 with vLLM, MATH-500 at 16 samples/problem:

| Preset | per arm | whole grid (9 arms) |
|---|---|---|
| math-1.5b | 10 to 15 min | about 2 h |
| math-7b | 30 to 45 min | 5 to 7 h |

Run the **smoke test** first (about 5 minutes). It exercises every code path on
20 problems and tells you the pipeline is wired before you spend compute units.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Install

`vllm` pins its own torch. The kernel never imports torch or vllm (generation runs
in a subprocess), so no runtime restart is needed.

In [ ]:
import subprocess, sys
PKGS = "vllm math-verify datasets hf_transfer pandas matplotlib"
r = subprocess.run(f"pip install -q uv && uv pip install --system -q {PKGS}", shell=True)
if r.returncode != 0:
    print("uv failed, falling back to pip")
    subprocess.run(f"pip install -q {PKGS}", shell=True, check=True)
print(subprocess.run([sys.executable, "-c", "import vllm, math_verify, datasets; print('vllm', vllm.__version__)"],
                     capture_output=True, text=True).stdout or "import check failed, see above")

## 2. Where results go

Results are written under `MyDrive/239_project/results/<run>/<dataset>/` so they
survive the session. Accept the Drive popup; if the path printed below starts with
`/content/239_project_runs` the results are on the VM disk and die with it.
Outside Colab this falls back to a local folder.

In [ ]:
import os
try:
    from google.colab import drive  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if ON_COLAB:
    # If the auth popup is dismissed this raises: results must land on Drive, not on the VM disk.
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/239_project"
    assert os.path.isdir("/content/drive/MyDrive"), "Drive did not mount; re-run this cell and accept the popup"
else:
    ROOT = os.path.abspath("239_project_runs")
RESULTS = f"{ROOT}/results"
os.makedirs(RESULTS, exist_ok=True)
WORK = "/content/239_project" if os.path.isdir("/content") else os.path.abspath("239_project_work")
os.makedirs(f"{WORK}/src", exist_ok=True)
os.chdir(WORK)
print("results ->", RESULTS)
print("work    ->", WORK)

## 3. Code

Three files, embedded from the repo's `src/`. Edit them in the repo and rebuild
the notebook with `python tools/build_notebook.py`; editing here works too for a
one-off run. `gen.py` prints its full CLI with `python src/gen.py --help`.

In [ ]:
%%writefile src/arith_check.py
"""Prototype: extract 'A op B = C' style arithmetic equalities from a math CoT and
check them exactly with Python Fractions. Returns (n_checked, n_wrong, details)."""
import re
from fractions import Fraction

NUM = r'-?\d{1,3}(?:,\d{3})+(?:\.\d+)?|-?\d+(?:\.\d+)?'
# Normalise common LaTeX/unicode operators to Python before matching.
_NORM = [
    (r'\\times|\\cdot|×|·|\*\*?', '*'),
    (r'\\div|÷', '/'),
    (r'\\left\(|\\right\)', lambda m: '(' if 'left' in m.group(0) else ')'),
    (r'\\frac\{([^{}]+)\}\{([^{}]+)\}', r'((\1)/(\2))'),
    (r'\\dfrac\{([^{}]+)\}\{([^{}]+)\}', r'((\1)/(\2))'),
    (r'\^\{([^{}]+)\}', r'^(\1)'),
    (r'\\,|\\!|\;|\\ ', ''),
    (r'\\%|%', ''),
    (r'\$', ''),
    (r'−|–', '-'),
]
def normalise(text):
    for pat, rep in _NORM:
        text = re.sub(pat, rep, text)
    return text

# LHS: a chain of numbers/parens/operators containing at least one operator; RHS: a single number.
LHS = r'(?<![\w.])(\(*\s*(?:%s)\s*\)*(?:\s*[-+*/^]\s*\(*\s*(?:%s)\s*\)*)+)' % (NUM, NUM)
# RHS may be followed by sentence punctuation, but not by more digits / a thousands group / a decimal tail
EQ = re.compile(LHS + r'\s*(?:=|\\approx|≈)\s*(%s)(?!\d|,\d{3}|\.\d)' % NUM)

def _to_frac(tok):
    return Fraction(tok.replace(',', ''))

def _eval(expr):
    # tokenise numbers, replace by Fraction literals, evaluate with ^ as power
    expr = expr.replace('^', '**')
    def sub(m): return 'Fraction("%s")' % m.group(0).replace(',', '')
    py = re.sub(NUM, sub, expr)
    if not re.fullmatch(r'[\sFraction("0-9.\-")+*/()\*]*', py):
        raise ValueError('unsafe')
    return eval(py, {'Fraction': Fraction, '__builtins__': {}})

def check(text, rel_tol=1e-6):
    text = normalise(text)
    n = wrong = 0
    details = []
    for m in EQ.finditer(text):
        lhs, rhs = m.group(1), m.group(2)
        try:
            v = _eval(lhs); r = _to_frac(rhs)
        except Exception:
            continue
        # tolerate rounding when RHS has decimals
        decimals = len(rhs.split('.')[1]) if '.' in rhs else 0
        ok = v == r or (decimals > 0 and abs(float(v) - float(r)) <= 0.5 * 10 ** -decimals + 1e-12) \
             or (r != 0 and abs(float(v - r) / float(r)) <= rel_tol)
        n += 1; wrong += (not ok)
        details.append((lhs.strip(), rhs, str(v), ok))
    return n, wrong, details

if __name__ == '__main__':
    samples = [
        "So 12 * 13 = 156 and 156 + 7 = 163.",
        "We get $3 \\times 17 = 51$, then $51 - 9 = 41$, so 41/2 = 20.5.",
        "Compute 1,234 + 5,678 = 6,912. Then 6912 / 8 = 864.",
        "Hence 2^{10} = 1024 and 2^5 = 32.",
        "So \\frac{3}{4} \\cdot 8 = 6.",
        "Wrong: 17 * 23 = 381. Also 100 - 37 = 73 is wrong too.",
        "Rounded: 22/7 = 3.14 is fine; 1/3 = 0.333.",
        "Equation x + 3 = 5 should not match; 2x = 4 should not match.",
        "Percent: 25% of 80 = 20 (not matched, fine). 15 * 0.2 = 3.",
        "Thus 4 + 4 = 8 = 2 * 4.",
    ]
    for s in samples:
        n, w, d = check(s)
        print(f"n={n} wrong={w} :: {s}")
        for x in d: print("    ", x)

In [ ]:
%%writefile src/gen.py
#!/usr/bin/env python
"""Sample one model under one or more tool conditions with vLLM and write JSONL.

Every model gets the same grid. Conditions:
  prompt_off   shared no-code prompt ("do not write code"). If the model writes a
               code block anyway, the harness intercepts it and injects the denial
               message, so nothing is ever executed and no output is fabricated.
               n_calls on these rows = non-compliance count.
  env_denied   the model's tool prompt (asks for code). Every code block is
               intercepted and answered with the denial message.
  tool_on      the model's tool prompt. Code is executed with `python -I` in a
               temp dir (NOT a sandbox: fine for Colab, not for a shared box).
               Calls past --max-calls get the budget message instead.
  plain_cot    optional. Shared plain CoT prompt, one pass, no interception at
               all: the condition prior work (CIR) used. Code the model writes is
               just text and it will fabricate its own ```output block.

Only `main()` imports vllm, so the helpers can be unit-tested on a laptop.

Example
  python gen.py --model GAIR/ToRL-1.5B --tag tool_rl \
      --conditions prompt_off,env_denied,tool_on --native-template torl \
      --dataset math500 --n-problems 100 --n 16 --max-tokens 3000 \
      --max-model-len 4096 --out results/math-1.5b
"""
import argparse
import json
import os
import random
import re
import subprocess
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import arith_check  # noqa: E402

# --------------------------------------------------------------------------- prompts
# Verified against the sources on 2026-10-13:
#   qwen_math_cot  Oat-Zero model card (sail/Qwen2.5-Math-*-Oat-Zero), also the
#                  Qwen2.5-Math CoT prompt. Shared tool-off prompt for every model.
#   qwen_math_tir  Qwen2.5-Math TIR prompt (same words ToRL uses).
#   torl           ToRL paper Fig. 3, verbatim, raw User/Assistant format.
#   cir            CIR paper (2505.24480) Fig. 5 text, verbatim. The paper does not
#                  say whether it sits in the system or user turn; system assumed.
TEMPLATES = {
    "qwen_math_cot": (
        "<|im_start|>system\nPlease reason step by step, and put your final answer "
        "within \\boxed{}.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    ),
    "qwen_math_nocode": (
        "<|im_start|>system\nPlease reason step by step, and put your final answer "
        "within \\boxed{}. Do not write code: solve the problem in natural language and "
        "arithmetic only.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    ),
    "qwen_math_tir": (
        "<|im_start|>system\nPlease integrate natural language reasoning with programs to "
        "solve the problem above, and put your final answer within \\boxed{}.<|im_end|>\n"
        "<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    ),
    "torl": (
        "A conversation between User and Assistant. The user asks a question, and the "
        "Assistant solves it.\nUser: Please integrate natural language reasoning with "
        "programs to solve the problem above, and put your final answer within \\boxed{}.\n"
        "{q}\nAssistant:"
    ),
    "cir": (
        "<|im_start|>system\nPlease solve the following problem step by step. During your "
        "reasoning process, if needed, you can choose to write python code to enhance your "
        "reasoning. The code executor will run your code and provide the execution results "
        "back to you to support your reasoning process. Please put the final answer within "
        "\\boxed{}.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    ),
}

# Injected in place of execution results. AdaTIR wording for the over-budget case;
# an honest "no interpreter" wording for prompt_off / env_denied (override: --denial-text).
DENIAL_BUDGET = ("Tool call budget exceeded; the code was not executed. "
                 "Please continue reasoning and give the final answer.")
DENIAL_UNAVAILABLE = ("No interpreter is available; the code was not executed. "
                      "Continue reasoning without running code and give the final answer.")
DENIAL = DENIAL_UNAVAILABLE

STOP_FENCE = "```\n"       # end of any fenced block
STOP_OUTPUT = "```output"  # model opens the output block itself
TEXT_STOPS = ["<|im_end|>", "<|endoftext|>", "\nUser:"]

CODE_RE = re.compile(r"```(?:python|py)[ \t]*\n(.*?)```", re.S)
FENCED_RE = re.compile(r"```.*?```", re.S)
CONDITIONS = ("prompt_off", "env_denied", "tool_on", "plain_cot")
LOOP_CONDITIONS = ("prompt_off", "env_denied", "tool_on")


def build_prompt(template: str, question: str) -> str:
    return TEMPLATES[template].replace("{q}", question)


def last_code_block(text: str):
    """Last closed ```python block in `text`, or None."""
    blocks = CODE_RE.findall(text)
    return blocks[-1] if blocks else None


def last_boxed(text: str):
    """Content of the last \\boxed{...} with balanced braces, or None."""
    i = text.rfind("\\boxed")
    if i < 0:
        return None
    j = text.find("{", i)
    if j < 0:
        return None
    depth = 0
    for k in range(j, len(text)):
        if text[k] == "{":
            depth += 1
        elif text[k] == "}":
            depth -= 1
            if depth == 0:
                return text[j + 1:k]
    return None


def strip_fenced(text: str) -> str:
    return FENCED_RE.sub(" ", text)


# --------------------------------------------------------------------------- grading
def make_grader():
    try:
        from math_verify import parse, verify  # type: ignore

        def grade(response: str, gold: str) -> bool:
            try:
                g = parse(gold if "\\boxed" in gold else f"${gold}$")
                p = parse(response)
                return bool(verify(g, p))
            except Exception:
                return False
        return grade, "math_verify"
    except ImportError:
        def norm(s):
            return re.sub(r"\s+|\\left|\\right|\\!|\\,|\$", "", s or "").replace("dfrac", "frac")

        def grade(response: str, gold: str) -> bool:
            b = last_boxed(response)
            return b is not None and norm(b) == norm(last_boxed(gold) or gold)
        return grade, "string"


# --------------------------------------------------------------------------- data
def load_problems(name: str, n_problems: int, seed: int):
    from datasets import load_dataset  # type: ignore
    if name == "math500":
        ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
        items = [dict(id=r["unique_id"], problem=r["problem"], answer=r["answer"],
                      subject=r["subject"], level=str(r["level"])) for r in ds]
    elif name == "aime24":
        ds = load_dataset("math-ai/aime24", split="test")
        items = [dict(id=f"aime24-{r['id']}", problem=r["problem"],
                      answer=last_boxed(r["solution"]) or r["solution"],
                      subject="aime24", level="aime") for r in ds]
    elif name == "aime25":
        ds = load_dataset("math-ai/aime25", split="test")
        items = [dict(id=f"aime25-{r['id']}", problem=r["problem"], answer=str(r["answer"]),
                      subject="aime25", level="aime") for r in ds]
    else:
        raise ValueError(f"unknown dataset {name}")
    if n_problems and n_problems < len(items):
        idx = sorted(random.Random(seed).sample(range(len(items)), n_problems))
        items = [items[i] for i in idx]
    return items


# --------------------------------------------------------------------------- executor
def execute(code: str, timeout: float = 10.0, max_chars: int = 2000):
    """Run `code` with python -I in a temp dir. Returns (output_text, errored)."""
    with tempfile.TemporaryDirectory() as d:
        path = Path(d) / "snippet.py"
        path.write_text(code)
        try:
            r = subprocess.run([sys.executable, "-I", str(path)], cwd=d, capture_output=True,
                               stdin=subprocess.DEVNULL, text=True, timeout=timeout)
            out, err = r.stdout, r.returncode != 0
            if err:
                tail = r.stderr.strip().splitlines()
                out = (out + "\n" + tail[-1]) if tail else out
        except subprocess.TimeoutExpired:
            out, err = f"Timeout: code ran longer than {timeout:.0f}s", True
    out = out.strip()
    if len(out) > max_chars:
        out = out[:max_chars] + "...[truncated]"
    return out, err


# --------------------------------------------------------------------------- tool loop
class Sample:
    __slots__ = ("pidx", "sidx", "prompt", "gen", "gen_tokens", "n_calls", "n_denied",
                 "n_exec_err", "n_injected", "rounds", "done", "truncated", "finish")

    def __init__(self, pidx, sidx, prompt):
        self.pidx, self.sidx, self.prompt = pidx, sidx, prompt
        self.gen = ""
        self.gen_tokens = 0
        self.n_calls = self.n_denied = self.n_exec_err = self.n_injected = self.rounds = 0
        self.done = self.truncated = False
        self.finish = None

    @property
    def context(self):
        return self.prompt + self.gen


def advance(s: Sample, text: str, n_tokens: int, finish_reason, stop_reason, cond: str,
            max_calls: int, executor=execute, denial=None):
    """Apply one generation chunk to sample `s`. Returns the text to inject (or None)."""
    s.gen += text
    s.gen_tokens += n_tokens
    s.rounds += 1
    if finish_reason == "length":
        s.done, s.truncated, s.finish = True, True, "length"
        return None
    if stop_reason not in (STOP_FENCE, STOP_OUTPUT):
        s.done, s.finish = True, "eos"
        return None
    code = last_code_block(text)
    if code is None:
        return None  # a non-python fence or a fabricated ```output: keep generating
    s.n_calls += 1
    if cond == "tool_on" and s.n_calls <= max_calls:
        result, err = executor(code)
        s.n_exec_err += int(err)
    else:
        result = DENIAL_BUDGET if cond == "tool_on" else (denial or DENIAL_UNAVAILABLE)
        s.n_denied += 1
    inj = ("\n" + result + "\n```\n") if stop_reason == STOP_OUTPUT else ("```output\n" + result + "\n```\n")
    s.gen += inj
    s.n_injected += 1
    return inj


TOOL_STOPS = (STOP_FENCE, STOP_OUTPUT)


def splice_round(pending, cond, args, pool):
    """pending: list of (sample, text, n_tokens, finish_reason, stop_reason) for one round.
    Executes tool_on code calls in parallel, then applies `advance` to every sample."""
    futures = {}
    for s, text, ntok, fin, stop in pending:
        if (cond == "tool_on" and stop in TOOL_STOPS and fin != "length"
                and s.n_calls + 1 <= args.max_calls):
            code = last_code_block(text)
            if code is not None:
                futures[id(s)] = pool.submit(execute, code, args.exec_timeout, args.max_output_chars)
    for s, text, ntok, fin, stop in pending:
        fut = futures.get(id(s))
        if fut is not None:
            ex = lambda code, f=fut: f.result()  # noqa: E731
        else:
            ex = lambda code: execute(code, args.exec_timeout, args.max_output_chars)  # noqa: E731
        advance(s, text, ntok, fin, stop, cond, args.max_calls, executor=ex,
                denial=getattr(args, "denial_text", None))


def run_tool_loop(generate, tokenize_len, samples, cond, args, pool):
    """Rounds 2..N. generate(prompts, stops, budgets, seeds) -> [(text, n_tokens, finish, stop_reason)]."""
    max_rounds = 2 * args.max_calls + 6
    while True:
        keep, budgets = [], []
        for s in samples:
            if s.done:
                continue
            remaining = min(args.max_tokens - s.gen_tokens,
                            args.max_model_len - tokenize_len(s.context) - 1)
            if remaining < 16:
                s.done, s.truncated, s.finish = True, True, "length"
            else:
                keep.append(s)
                budgets.append(remaining)
        if not keep:
            break
        final = any(s.rounds >= max_rounds for s in keep)
        stops = TEXT_STOPS if final else TEXT_STOPS + list(TOOL_STOPS)
        print(f"  round: {len(keep)} active, final={final}", flush=True)
        seeds = [(args.seed * 1_000_003 + s.pidx * 4096 + s.sidx * 64 + s.rounds) % (2 ** 31) for s in keep]
        outs = generate([s.context for s in keep], stops, budgets, seeds)
        splice_round([(s,) + tuple(o) for s, o in zip(keep, outs)], cond, args, pool)


# --------------------------------------------------------------------------- records
def to_record(s: Sample, item, cond, template, model, tag, grade):
    text = s.gen
    fake_outputs = max(0, text.count(STOP_OUTPUT) - s.n_injected)
    prose = strip_fenced(text)
    a_n, a_wrong, _ = arith_check.check(prose)
    return dict(
        model=model, tag=tag, condition=cond, template=template,
        problem_id=item["id"], subject=item.get("subject"), level=item.get("level"),
        sample_idx=s.sidx, correct=bool(grade(text, item["answer"])), gold=item["answer"],
        boxed=last_boxed(text), boxed_present=last_boxed(text) is not None,
        truncated=s.truncated, finish=s.finish, gen_tokens=s.gen_tokens, rounds=s.rounds,
        code_emitted="```python" in text or "```py" in text, fake_outputs=fake_outputs,
        n_calls=s.n_calls, n_denied=s.n_denied, n_exec_err=s.n_exec_err,
        arith_n=a_n, arith_wrong=a_wrong, text=text,
    )


# --------------------------------------------------------------------------- main
def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", required=True)
    p.add_argument("--revision", default=None)
    p.add_argument("--tag", required=True, help="role label: base | tool_rl | text_rl | ...")
    p.add_argument("--conditions", default="prompt_off")
    p.add_argument("--shared-template", default="qwen_math_nocode",
                   help="no-code prompt used for prompt_off, same for every model")
    p.add_argument("--native-template", default="qwen_math_tir",
                   help="the model's tool prompt, used for env_denied and tool_on")
    p.add_argument("--plain-template", default="qwen_math_cot", help="prompt for plain_cot")
    p.add_argument("--denial-text", default=None,
                   help="override the message injected for prompt_off / env_denied")
    p.add_argument("--dataset", default="math500", choices=["math500", "aime24", "aime25"])
    p.add_argument("--n-problems", type=int, default=0, help="0 = all")
    p.add_argument("--n", type=int, default=16, help="samples per problem")
    p.add_argument("--max-tokens", type=int, default=3000)
    p.add_argument("--max-model-len", type=int, default=4096)
    p.add_argument("--allow-long", action="store_true",
                   help="override max_position_embeddings so max-model-len can exceed the config")
    p.add_argument("--temperature", type=float, default=0.6)
    p.add_argument("--top-p", type=float, default=0.95)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--max-calls", type=int, default=4)
    p.add_argument("--exec-timeout", type=float, default=10.0)
    p.add_argument("--exec-workers", type=int, default=8)
    p.add_argument("--max-output-chars", type=int, default=2000)
    p.add_argument("--gpu-mem", type=float, default=0.90)
    p.add_argument("--dtype", default="auto")
    p.add_argument("--out", required=True, help="results directory")
    p.add_argument("--overwrite", action="store_true", help="redo finished arms and discard partial files")
    p.add_argument("--chunk", type=int, default=100,
                   help="problems per checkpoint; records are appended to <arm>.jsonl.partial after each "
                        "chunk and a restart resumes from there (0 = whole arm in one go)")
    return p.parse_args(argv)


# --------------------------------------------------------------------------- checkpointing
# An arm is "done" when its .meta.json exists (written last). While an arm runs, each chunk of
# problems is appended to <arm>.jsonl.partial, so a disconnect loses at most one chunk. The
# final .jsonl is written atomically (tmp + rename) and the partial file is removed.
def partial_path(path: Path) -> Path:
    return Path(str(path) + ".partial")


def meta_path(path: Path) -> Path:
    return path.with_suffix(".meta.json")


def read_partial(path: Path, n: int):
    """Records already on disk for this arm. Keeps only problems with all n sample_idx present
    (last write wins), so a torn chunk is simply redone. Returns (records, finished problem ids)."""
    pp = partial_path(path)
    if not pp.exists():
        return [], set()
    by_pid = {}
    with open(pp) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue  # torn last line from a crash mid-write
            by_pid.setdefault(r["problem_id"], {})[r["sample_idx"]] = r
    done = {pid for pid, d in by_pid.items() if len(d) == n}
    recs = [by_pid[pid][k] for pid in done for k in sorted(by_pid[pid])]
    return recs, done


def plan_chunks(items, done_ids, chunk):
    """Remaining (global_index, item) pairs, grouped into chunks. Global indices are kept so
    per-sample seeds do not depend on chunking or on what was already finished."""
    remaining = [(i, it) for i, it in enumerate(items) if it["id"] not in done_ids]
    size = chunk if chunk > 0 else max(1, len(remaining))
    return [remaining[s:s + size] for s in range(0, len(remaining), size)]


def write_jsonl(path: Path, recs, mode="a"):
    # A crash mid-write leaves a torn last line with no newline; start the append on a fresh line
    # so the torn line stays one (ignorable) bad record instead of swallowing the next good one.
    if mode == "a" and path.exists() and path.stat().st_size > 0:
        with open(path, "rb") as f:
            f.seek(-1, os.SEEK_END)
            needs_nl = f.read(1) != b"\n"
        if needs_nl:
            with open(path, "a") as f:
                f.write("\n")
    with open(path, mode) as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass  # FUSE mounts such as Google Drive may not support fsync


def write_jsonl_atomic(path: Path, recs):
    tmp = Path(str(path) + ".tmp")
    write_jsonl(tmp, recs, mode="w")
    os.replace(tmp, path)


def main(argv=None):
    args = parse_args(argv)
    conds = [c.strip() for c in args.conditions.split(",") if c.strip()]
    for c in conds:
        if c not in CONDITIONS:
            sys.exit(f"unknown condition {c}; choose from {CONDITIONS}")
    out_dir = Path(args.out) / args.dataset
    out_dir.mkdir(parents=True, exist_ok=True)

    def out_path(cond, template):
        return out_dir / f"{args.tag}__{cond}__{template}.jsonl"

    def template_for(cond):
        if cond == "prompt_off":
            return args.shared_template
        if cond == "plain_cot":
            return args.plain_template
        return args.native_template

    todo = [(c, template_for(c)) for c in conds
            if args.overwrite or not meta_path(out_path(c, template_for(c))).exists()]
    if not todo:
        print("all conditions already done, nothing to do (use --overwrite to redo)")
        return
    print(f"gen.py starting: {args.tag} {args.model}; todo={todo}", flush=True)
    print(f"loading {args.dataset} from the HF hub...", flush=True)
    items = load_problems(args.dataset, args.n_problems, args.seed)
    print("loading grader (math_verify)...", flush=True)
    grade, grader_name = make_grader()
    print(f"{len(items)} problems from {args.dataset}; grader={grader_name}", flush=True)
    print("importing torch + vllm (first time: a minute or two)...", flush=True)

    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
    if args.allow_long:
        os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "1"
    import torch  # type: ignore
    from vllm import LLM, SamplingParams  # type: ignore
    dtype = args.dtype
    if dtype == "auto":
        major = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 8
        dtype = "bfloat16" if major >= 8 else "float16"
    kw = dict(model=args.model, dtype=dtype, max_model_len=args.max_model_len, seed=args.seed,
              gpu_memory_utilization=args.gpu_mem, enable_prefix_caching=True)
    if args.revision:
        kw["revision"] = args.revision
    if args.allow_long:
        kw["hf_overrides"] = {"max_position_embeddings": args.max_model_len}
    t0 = time.time()
    llm = LLM(**kw)
    tok = llm.get_tokenizer()
    print(f"loaded {args.model} ({dtype}) in {time.time() - t0:.0f}s", flush=True)

    def tokenize_len(text):
        return len(tok(text, add_special_tokens=False)["input_ids"])

    def sp(n, max_tokens, stops, seed=None):
        return SamplingParams(n=n, temperature=args.temperature, top_p=args.top_p,
                              seed=args.seed if seed is None else seed,
                              max_tokens=max_tokens, stop=stops, include_stop_str_in_output=True)

    def generate(prompts, stops, budgets, seeds):
        params = [sp(1, b, stops, seed) for b, seed in zip(budgets, seeds)]
        res = llm.generate(prompts, params, use_tqdm=False)
        outs = []
        for r in res:
            o = r.outputs[0]
            outs.append((o.text, len(o.token_ids), o.finish_reason, o.stop_reason))
        return outs

    def run_arm(part, cond, template):
        """Generate every sample for the (global_index, item) pairs in `part`; returns Samples."""
        prompts = [build_prompt(template, it["problem"]) for _, it in part]
        samples = []
        if cond == "plain_cot":
            res = llm.generate(prompts, sp(args.n, args.max_tokens, TEXT_STOPS), use_tqdm=True)
            for (pidx, _), prompt, r in zip(part, prompts, res):
                for sidx, o in enumerate(r.outputs):
                    s = Sample(pidx, sidx, prompt)
                    advance(s, o.text, len(o.token_ids), o.finish_reason, None, cond, 0)
                    if o.finish_reason == "length":
                        s.truncated, s.finish = True, "length"
                    samples.append(s)
            return samples
        res = llm.generate(prompts, sp(args.n, args.max_tokens, TEXT_STOPS + list(TOOL_STOPS)), use_tqdm=True)
        first = []
        for (pidx, _), prompt, r in zip(part, prompts, res):
            for sidx, o in enumerate(r.outputs):
                s = Sample(pidx, sidx, prompt)
                samples.append(s)
                first.append((s, o.text, len(o.token_ids), o.finish_reason, o.stop_reason))
        pool = ThreadPoolExecutor(max_workers=args.exec_workers)
        splice_round(first, cond, args, pool)
        run_tool_loop(generate, tokenize_len, samples, cond, args, pool)
        pool.shutdown(wait=True)
        return samples

    order = {it["id"]: i for i, it in enumerate(items)}
    for cond, template in todo:
        t1 = time.time()
        path = out_path(cond, template)
        if args.overwrite:
            partial_path(path).unlink(missing_ok=True)
        kept, _ = read_partial(path, args.n)
        kept = [r for r in kept if r["problem_id"] in order]
        done_ids = {r["problem_id"] for r in kept}
        chunks = plan_chunks(items, done_ids, args.chunk)
        print(f"== {args.tag} / {cond} / {template}: {len(items)} problems x {args.n} samples"
              + (f"; resuming, {len(done_ids)} problems already on disk" if done_ids else "")
              + f"; {len(chunks)} chunk(s) to run", flush=True)
        new_recs = []
        for ci, part in enumerate(chunks, 1):
            tc = time.time()
            samples = run_arm(part, cond, template)
            recs = [to_record(s, items[s.pidx], cond, template, args.model, args.tag, grade) for s in samples]
            write_jsonl(partial_path(path), recs)
            new_recs += recs
            print(f"   chunk {ci}/{len(chunks)}: {len(part)} problems, pass@1="
                  f"{sum(r['correct'] for r in recs) / max(1, len(recs)):.3f}, "
                  f"{(time.time() - tc) / 60:.1f} min, checkpointed", flush=True)
        gen_s = time.time() - t1
        recs = kept + new_recs
        recs.sort(key=lambda r: (order[r["problem_id"]], r["sample_idx"]))
        write_jsonl_atomic(path, recs)
        acc = sum(r["correct"] for r in recs) / max(1, len(recs))
        meta = dict(vars(args), condition=cond, template=template, dtype=dtype, grader=grader_name,
                    n_problems=len(items), n_records=len(recs), n_resumed=len(done_ids), pass1=acc,
                    gen_seconds=gen_s, total_gen_tokens=sum(r["gen_tokens"] for r in recs))
        meta_path(path).write_text(json.dumps(meta, indent=1))
        partial_path(path).unlink(missing_ok=True)
        print(f"   pass@1={acc:.3f}  truncated={sum(r['truncated'] for r in recs) / len(recs):.2%}  "
              f"tokens={meta['total_gen_tokens'] / 1e6:.2f}M  {gen_s / 60:.1f} min -> {path}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/analyze.py
"""Load gen.py results, compute pass@k, arithmetic error rates, and plot.

Usage in a notebook:
    import analyze as an
    df = an.load_results("results/math-1.5b/math500")
    an.summary(df)                 # one row per (tag, condition)
    an.plot_passk(df)              # pass@k curves, one panel per condition
    an.verdict(df)                 # atrophy / internalization / overlap readout
"""
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

ROLE_ORDER = ["base", "text_rl", "tool_rl"]
# dataviz reference palette, categorical slots 1-3 (CVD-validated): blue, orange, aqua
ROLE_COLOR = {"base": "#2a78d6", "tool_rl": "#eb6834", "text_rl": "#1baf7a"}
COND_ORDER = ["prompt_off", "env_denied", "tool_on", "plain_cot"]
COND_STYLE = {"prompt_off": "-", "env_denied": "--", "tool_on": ":", "plain_cot": "-."}
COND_MARK = {"prompt_off": "o", "env_denied": "s", "tool_on": "^", "plain_cot": "D"}


def load_results(results_dir) -> pd.DataFrame:
    paths = sorted(Path(results_dir).glob("*.jsonl"))
    if not paths:
        raise FileNotFoundError(f"no .jsonl under {results_dir}")
    frames = []
    for p in paths:
        rows = [json.loads(line) for line in open(p)]
        f = pd.DataFrame(rows)
        f["file"] = p.name
        frames.append(f)
    df = pd.concat(frames, ignore_index=True)
    df["arm"] = df["tag"] + " / " + df["condition"]
    return df


def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased estimator from Chen et al. 2021."""
    if k > n:
        return float("nan")
    if n - c < k:
        return 1.0
    return 1.0 - math.exp(math.lgamma(n - c + 1) - math.lgamma(n - c - k + 1)
                          - math.lgamma(n + 1) + math.lgamma(n - k + 1))


def ks_for(n: int):
    ks = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
    return [k for k in ks if k <= n]


def per_problem(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["tag", "condition", "problem_id"])["correct"]
    out = g.agg(n="count", c="sum").reset_index()
    return out


def passk_table(df: pd.DataFrame) -> pd.DataFrame:
    """Rows: (tag, condition); columns: pass@k means over problems."""
    pp = per_problem(df)
    rows = []
    for (tag, cond), grp in pp.groupby(["tag", "condition"]):
        n = int(grp["n"].min())
        row = dict(tag=tag, condition=cond, n_problems=len(grp), n_samples=n)
        for k in ks_for(n):
            row[f"pass@{k}"] = float(np.mean([pass_at_k(int(r.n), int(r.c), k) for r in grp.itertuples()]))
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["condition", "tag"]).reset_index(drop=True)


def bootstrap_passk(df: pd.DataFrame, tag: str, cond: str, k: int, iters: int = 1000, seed: int = 0):
    """Percentile bootstrap over problems. Returns (mean, lo, hi)."""
    pp = per_problem(df)
    grp = pp[(pp.tag == tag) & (pp.condition == cond)]
    vals = np.array([pass_at_k(int(r.n), int(r.c), k) for r in grp.itertuples()])
    if len(vals) == 0:
        return (float("nan"),) * 3
    rng = np.random.default_rng(seed)
    boots = rng.choice(vals, size=(iters, len(vals)), replace=True).mean(axis=1)
    return float(vals.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def paired_diff(df: pd.DataFrame, a: tuple, b: tuple, k: int, iters: int = 1000, seed: int = 0):
    """Bootstrap CI of pass@k(a) - pass@k(b), paired by problem. a, b = (tag, condition)."""
    pp = per_problem(df)
    A = pp[(pp.tag == a[0]) & (pp.condition == a[1])].set_index("problem_id")
    B = pp[(pp.tag == b[0]) & (pp.condition == b[1])].set_index("problem_id")
    ids = A.index.intersection(B.index)
    if len(ids) == 0:
        return (float("nan"),) * 3
    da = np.array([pass_at_k(int(A.loc[i, "n"]), int(A.loc[i, "c"]), k) for i in ids])
    db = np.array([pass_at_k(int(B.loc[i, "n"]), int(B.loc[i, "c"]), k) for i in ids])
    d = da - db
    rng = np.random.default_rng(seed)
    boots = rng.choice(d, size=(iters, len(d)), replace=True).mean(axis=1)
    return float(d.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def summary(df: pd.DataFrame) -> pd.DataFrame:
    """Headline table: accuracy, pass@kmax, failure modes, arithmetic error rate."""
    pk = passk_table(df)
    rows = []
    for (tag, cond), g in df.groupby(["tag", "condition"]):
        n = int(g.groupby("problem_id").size().min())
        kmax = ks_for(n)[-1]
        pkrow = pk[(pk.tag == tag) & (pk.condition == cond)].iloc[0]
        arith_n = int(g["arith_n"].sum())
        rows.append(dict(
            tag=tag, condition=cond, template=g["template"].iloc[0],
            problems=g["problem_id"].nunique(), samples=n,
            **{"pass@1": round(pkrow["pass@1"], 3), f"pass@{kmax}": round(pkrow[f"pass@{kmax}"], 3)},
            trunc_pct=round(100 * g["truncated"].mean(), 1),
            no_boxed_pct=round(100 * (~g["boxed_present"]).mean(), 1),
            code_pct=round(100 * g["code_emitted"].mean(), 1),
            fake_out_pct=round(100 * (g["fake_outputs"] > 0).mean(), 1),
            calls_mean=round(g["n_calls"].mean(), 2), denied_mean=round(g["n_denied"].mean(), 2),
            exec_err_mean=round(g["n_exec_err"].mean(), 2),
            arith_err_pct=round(100 * g["arith_wrong"].sum() / arith_n, 2) if arith_n else float("nan"),
            arith_checked=arith_n, arith_coverage_pct=round(100 * (g["arith_n"] > 0).mean(), 1),
            tokens_mean=round(g["gen_tokens"].mean()),
        ))
    out = pd.DataFrame(rows)
    out["_r"] = out["tag"].map({t: i for i, t in enumerate(ROLE_ORDER)}).fillna(9)
    return out.sort_values(["condition", "_r"]).drop(columns="_r").reset_index(drop=True)


def _plot_lines(df, pk, ax, ci, iters, label_cond=True):
    for _, r in pk.iterrows():
        tag, cond = r["tag"], r["condition"]
        ks = ks_for(int(r["n_samples"]))
        ys = [r[f"pass@{k}"] for k in ks]
        color = ROLE_COLOR.get(tag, "#52514e")
        ax.plot(ks, ys, COND_STYLE.get(cond, "-"), color=color, lw=2,
                marker=COND_MARK.get(cond, "o"), ms=6,
                label=f"{tag} / {cond}" if label_cond else tag)
        if ci:
            lo, hi = [], []
            for k in ks:
                _, l, h = bootstrap_passk(df, tag, cond, k, iters=iters)
                lo.append(l); hi.append(h)
            ax.fill_between(ks, lo, hi, color=color, alpha=0.12, lw=0)
    kmax = ks_for(int(pk["n_samples"].max()))
    ax.set_xscale("log", base=2)
    ax.set_xticks(kmax)
    ax.set_xticklabels([str(k) for k in kmax])
    ax.set_xlabel("k (samples)")
    ax.set_ylim(0, 1)
    ax.grid(True, color="#e6e5e1", lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower right")


def plot_passk(df: pd.DataFrame, title=None, ci=True, iters=300, facet=True):
    """One panel per condition (three models per panel), or one panel with every arm."""
    import matplotlib.pyplot as plt
    pk = passk_table(df)
    conds = [c for c in COND_ORDER if c in set(pk.condition)] + \
            sorted(set(pk.condition) - set(COND_ORDER))
    if not facet or len(conds) == 1:
        fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
        _plot_lines(df, pk, ax, ci, iters)
        ax.set_ylabel("pass@k")
        ax.set_title(title or f"pass@k, {df['problem_id'].nunique()} problems", loc="left")
        return fig
    fig, axes = plt.subplots(1, len(conds), figsize=(4.6 * len(conds), 4.2), dpi=120, sharey=True)
    for ax, cond in zip(axes, conds):
        sub = pk[pk.condition == cond].copy()
        sub["_r"] = sub["tag"].map({t: i for i, t in enumerate(ROLE_ORDER)}).fillna(9)
        _plot_lines(df[df.condition == cond], sub.sort_values("_r"), ax, ci, iters, label_cond=False)
        ax.set_title(cond, loc="left")
    axes[0].set_ylabel("pass@k")
    fig.suptitle(title or f"pass@k, {df['problem_id'].nunique()} problems", x=0.01, ha="left")
    fig.tight_layout()
    return fig


def verdict(df: pd.DataFrame, cond: str = "env_denied", iters: int = 1000):
    """Print the three-way readout from the plan for one condition (same prompt, all models)."""
    pk = passk_table(df)
    sub = pk[pk.condition == cond]
    tags = set(sub.tag)
    need = {"base", "text_rl", "tool_rl"}
    if not need <= tags:
        print(f"[{cond}] need base, text_rl, tool_rl; have {sorted(tags)}")
        return
    n = int(sub.n_samples.min())
    kmax = ks_for(n)[-1]
    print(f"[{cond}] {sub.n_problems.min()} problems, n={n}, kmax={kmax}")
    for k in (1, kmax):
        for a, b in ((("tool_rl", cond), ("text_rl", cond)), (("tool_rl", cond), ("base", cond)),
                     (("text_rl", cond), ("base", cond))):
            m, lo, hi = paired_diff(df, a, b, k, iters=iters)
            sig = "" if lo <= 0 <= hi else "  *"
            print(f"  pass@{k:<4} {a[0]:>7} - {b[0]:<7} = {m:+.3f}  [{lo:+.3f}, {hi:+.3f}]{sig}")
    g = df[df.condition == cond].groupby("tag")
    er = (g["arith_wrong"].sum() / g["arith_n"].sum() * 100).round(2)
    print("  arithmetic error rate (%):", er.to_dict())
    d1 = paired_diff(df, ("tool_rl", cond), ("text_rl", cond), 1, iters=iters)
    dk = paired_diff(df, ("tool_rl", cond), ("text_rl", cond), kmax, iters=iters)
    if dk[2] < 0:
        print("  -> tool_rl below text_rl at high k: ATROPHY signature")
    elif d1[1] > 0 and dk[1] > 0:
        print("  -> tool_rl above text_rl at pass@1 and high k: INTERNALIZATION signature")
    elif d1[1] <= 0 <= d1[2] and dk[1] <= 0 <= dk[2]:
        print("  -> tool_rl and text_rl overlap at every k: tool added nothing beyond RL (kill criterion)")
    else:
        print("  -> mixed: pass@1 gain without a high-k deficit, or vice versa; look at the curve")

## 4. Configure

Presets name a (base, text-RL, tool-RL) trio from one base model. Each model also
carries its tool prompt: ToRL's own (paper Fig. 3) for ToRL, the Qwen2.5-Math TIR
prompt (same instruction sentence, chat format) for base and Oat-Zero, CIR's Fig. 5
text for the CIR checkpoint. Set every `native` to the same key if you want one
identical tool prompt for all three.

| Preset | base | text-RL (control) | tool-RL |
|---|---|---|---|
| `math-1.5b` | Qwen/Qwen2.5-Math-1.5B | sail/Qwen2.5-Math-1.5B-Oat-Zero | GAIR/ToRL-1.5B |
| `math-7b` | Qwen/Qwen2.5-Math-7B | sail/Qwen2.5-Math-7B-Oat-Zero | RUC-AIBOX/Qwen-Math-7B-CIR (the Fig. 6 model) |
| `math-7b-torl` | Qwen/Qwen2.5-Math-7B | sail/Qwen2.5-Math-7B-Oat-Zero | GAIR/ToRL-7B |
| `general-7b-zerotir` | Qwen/Qwen2.5-7B | hkust-nlp/Qwen-2.5-7B-SimpleRL-Zoo | htxu91/zero-tir-7b-550step (tool prompt unverified) |

Context is capped at 4096 tokens for the Qwen2.5-Math trios because that is the
base model's and Oat-Zero's native context (CIR evaluated at 16k by extrapolating;
`ALLOW_LONG=True` reproduces that). Truncation rate is reported either way.

`PLAIN_COT=True` adds the fourth, non-intercepting condition for a tie to CIR's
numbers. `DENIAL_TEXT` overrides the message injected under `prompt_off` and
`env_denied`.

In [ ]:
PRESETS = {
    "math-1.5b": dict(max_model_len=4096, models={
        "base":    dict(model="Qwen/Qwen2.5-Math-1.5B",            native="qwen_math_tir"),
        "text_rl": dict(model="sail/Qwen2.5-Math-1.5B-Oat-Zero",   native="qwen_math_tir"),
        "tool_rl": dict(model="GAIR/ToRL-1.5B",                     native="torl"),
    }),
    "math-7b": dict(max_model_len=4096, models={
        "base":    dict(model="Qwen/Qwen2.5-Math-7B",              native="qwen_math_tir"),
        "text_rl": dict(model="sail/Qwen2.5-Math-7B-Oat-Zero",     native="qwen_math_tir"),
        "tool_rl": dict(model="RUC-AIBOX/Qwen-Math-7B-CIR",        native="cir"),
    }),
    "math-7b-torl": dict(max_model_len=4096, models={
        "base":    dict(model="Qwen/Qwen2.5-Math-7B",              native="qwen_math_tir"),
        "text_rl": dict(model="sail/Qwen2.5-Math-7B-Oat-Zero",     native="qwen_math_tir"),
        "tool_rl": dict(model="GAIR/ToRL-7B",                       native="torl"),
    }),
    "general-7b-zerotir": dict(max_model_len=8192, models={
        "base":    dict(model="Qwen/Qwen2.5-7B",                   native="qwen_math_tir"),
        "text_rl": dict(model="hkust-nlp/Qwen-2.5-7B-SimpleRL-Zoo", native="qwen_math_tir"),
        "tool_rl": dict(model="htxu91/zero-tir-7b-550step",         native="qwen_math_tir"),
    }),
}

PRESET = "math-1.5b"       # start here; "math-7b" once the 1.5B run is clean
DATASET = "math500"        # math500 | aime24 | aime25
SMOKE = True               # True: 20 problems x 4 samples x 1024 tokens, ~10 min. False: the real run.

N_PROBLEMS = 20 if SMOKE else 0        # 0 = all 500 (AIME sets have 30)
N_SAMPLES = 4 if SMOKE else 16         # raise to 32 or 64 for a sharper high-k tail
MAX_TOKENS = 1024 if SMOKE else 3000
TEMPERATURE, TOP_P, SEED = 0.6, 0.95, 0   # CIR Fig. 6 settings
MAX_CALLS = 4
ALLOW_LONG = False         # True + MAX_MODEL_LEN=16384 mimics CIR's 16k budget on 4k-native models
MAX_MODEL_LEN = PRESETS[PRESET]["max_model_len"]
SHARED_TEMPLATE = "qwen_math_nocode"   # prompt_off prompt, identical for every model
PLAIN_COT = False          # True adds the non-intercepting plain_cot arm (CIR's condition)
DENIAL_TEXT = None         # None = "No interpreter is available; the code was not executed. ..."
GPU_MEM = 0.90
CHUNK = 100                # problems per checkpoint; a disconnect loses at most this much work
OVERWRITE = False          # True regenerates arms that are already on Drive

CONDITIONS = ["prompt_off", "env_denied", "tool_on"] + (["plain_cot"] if PLAIN_COT else [])
RUN_NAME = f"{PRESET}{'-smoke' if SMOKE else ''}"
OUT = f"{RESULTS}/{RUN_NAME}"
print(RUN_NAME, "->", OUT)

## 5. Run

One cell per checkpoint, below. Each cell first looks at what Drive already holds
for that model:

| On Drive | What the cell does |
|---|---|
| `.meta.json` for the arm | finished: loaded as is, nothing generated |
| `.jsonl.partial` only | interrupted: resumes from the last checkpointed chunk of `CHUNK` problems |
| nothing | generated from scratch |

So after a disconnect, reconnect, re-run cells 1 to 4 (install, Drive, code,
config), then re-run the model cell that was going. Finished arms and finished
chunks are never redone unless `OVERWRITE = True`. The three cells can run in any
order, one at a time (one GPU). Each one ends with a quick pass@1 table read back
from Drive. The first run of each model spends a few minutes downloading weights.

`samples` in that table is rows, which is problems x `N_SAMPLES`. If it is a
multiple of what you expect, a file from an older run with a different prompt
template sits in the same folder and loads as the same arm. `status` flags this;
`move_stale()` moves such files into `old/`.

In [ ]:
import json, os, shutil, subprocess, sys, time, importlib
from pathlib import Path
sys.path.insert(0, "src")


def arm_files(tag):
    spec = PRESETS[PRESET]["models"][tag]
    out = Path(OUT) / DATASET
    rows = []
    for cond in CONDITIONS:
        tmpl = SHARED_TEMPLATE if cond == "prompt_off" else ("qwen_math_cot" if cond == "plain_cot" else spec["native"])
        stem = out / f"{tag}__{cond}__{tmpl}"
        rows.append((cond, stem.with_suffix(".jsonl"), stem.with_suffix(".meta.json"), Path(str(stem) + ".jsonl.partial")))
    return rows


def partial_problems(partial):
    pids = set()
    for line in open(partial):
        try:
            pids.add(json.loads(line)["problem_id"])
        except (json.JSONDecodeError, KeyError):
            pass  # torn last line
    return len(pids)


DATA_SUFFIXES = (".jsonl", ".meta.json", ".jsonl.partial", ".jsonl.tmp")


def expected_stems():
    return {jsonl.stem for tag in PRESETS[PRESET]["models"] for _, jsonl, _, _ in arm_files(tag)}


def stale_files():
    # Data files in the run folder that the current config does not produce (an older template or
    # model), plus arms written by an older gen.py (no "chunk" key in meta). They inflate the
    # tables (two files load as one arm) and block regeneration (gen.py sees the arm as done).
    out = Path(OUT) / DATASET
    exp = expected_stems()
    stale = []
    for f in sorted(out.iterdir()) if out.is_dir() else []:
        if not f.is_file() or not f.name.endswith(DATA_SUFFIXES):
            continue
        stem = f.name.split(".")[0]
        if stem not in exp:
            stale.append((f, "not an arm of the current config"))
        elif f.name.endswith(".meta.json") and "chunk" not in json.load(open(f)):
            stale.append((f, "written by an older notebook version"))
            stale.append((out / f"{stem}.jsonl", "written by an older notebook version"))
    return stale


def move_stale():
    # Quarantine stale files into <run>/<dataset>/old/ (nothing is deleted).
    out = Path(OUT) / DATASET
    stale = stale_files()
    if not stale:
        print("nothing stale in", out); return
    (out / "old").mkdir(exist_ok=True)
    for f, why in stale:
        if f.exists():
            shutil.move(str(f), str(out / "old" / f.name))
            print(f"moved {f.name} -> old/   ({why})")


def load_run():
    # Results of the current config only; files from other runs in the folder are reported and ignored.
    import analyze as an
    importlib.reload(an)
    df = an.load_results(f"{OUT}/{DATASET}")
    exp = {f"{stem}.jsonl" for stem in expected_stems()}
    stray = sorted(set(df.file) - exp)
    if stray:
        print("ignoring", len(stray), "file(s) not in the current config:", ", ".join(stray))
        print("run move_stale() to quarantine them")
    return df[df.file.isin(exp)].copy()


def status(tag):
    # What Drive holds for this model. Returns how many arms still need generation.
    missing = 0
    for cond, jsonl, meta, partial in arm_files(tag):
        if meta.exists():
            m = json.load(open(meta))
            note = f"done     pass@1={m['pass1']:.3f}  n={m['n']}  problems={m['n_problems']}  {m['gen_seconds'] / 60:.0f} min"
            if m["n"] != N_SAMPLES or (N_PROBLEMS and m["n_problems"] != N_PROBLEMS):
                note += "   <- written with a different N_SAMPLES/N_PROBLEMS; set OVERWRITE=True or change RUN_NAME"
            if "chunk" not in m:
                note += "   <- older notebook version; run move_stale(), then re-run this cell"
        elif partial.exists():
            note = f"partial  {partial_problems(partial)} problems checkpointed, will resume"
            missing += 1
        else:
            note = "missing"
            missing += 1
        print(f"  {tag:8s} {cond:11s} {note}")
    n_stale = len({f for f, _ in stale_files()})
    if n_stale:
        print(f"  ! {n_stale} stale file(s) from another run in this folder: run move_stale() first")
    return missing


def show(tag):
    import analyze as an
    try:
        df = load_run()
    except FileNotFoundError:
        print("no results yet"); return
    d = df[df.tag == tag]
    if d.empty:
        print(f"no finished arms for {tag} yet"); return
    tbl = d.groupby("condition").agg(samples=("correct", "size"), problems=("problem_id", "nunique"),
                                      pass1=("correct", "mean"), trunc=("truncated", "mean"),
                                      calls=("n_calls", "mean"), denied=("n_denied", "mean")).round(3)
    tbl = tbl.reindex([c for c in an.COND_ORDER if c in tbl.index])
    try:
        display(tbl)
    except NameError:
        print(tbl.to_string())


def stream(cmd):
    # Relay the subprocess's stdout+stderr through the kernel: a line per tqdm refresh instead of
    # a carriage-return bar, so progress shows even when Colab does not mirror child output.
    # The child runs in its own process group; the stop button kills the whole group (gen.py and
    # vLLM's engine-core process), so nothing is left holding the GPU.
    import signal
    env = dict(os.environ, TQDM_MININTERVAL="20", PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                            start_new_session=True)
    try:
        while True:
            chunk = proc.stdout.read1(65536)
            if not chunk:
                break
            print(chunk.decode("utf-8", "replace").replace("\r", "\n"), end="", flush=True)
        return proc.wait()
    except BaseException:
        for sig, wait in ((signal.SIGTERM, 15), (signal.SIGKILL, 5)):
            try:
                os.killpg(proc.pid, sig)
                proc.wait(timeout=wait)
                break
            except subprocess.TimeoutExpired:
                continue
            except ProcessLookupError:
                break
        print(f"\n[interrupted] killed pid {proc.pid} and its children; finished arms and chunks are kept",
              flush=True)
        raise


def run_model(tag):
    spec = PRESETS[PRESET]["models"][tag]
    print(f"[{tag}] {spec['model']}  ->  {OUT}/{DATASET}")
    if status(tag) == 0 and not OVERWRITE:
        print(f"[{tag}] every arm is already on Drive, loading instead of regenerating")
    else:
        cmd = [sys.executable, "src/gen.py",
               "--model", spec["model"], "--tag", tag,
               "--conditions", ",".join(CONDITIONS),
               "--shared-template", SHARED_TEMPLATE, "--native-template", spec["native"],
               "--dataset", DATASET, "--n-problems", str(N_PROBLEMS), "--n", str(N_SAMPLES),
               "--max-tokens", str(MAX_TOKENS), "--max-model-len", str(MAX_MODEL_LEN),
               "--temperature", str(TEMPERATURE), "--top-p", str(TOP_P), "--seed", str(SEED),
               "--max-calls", str(MAX_CALLS), "--gpu-mem", str(GPU_MEM), "--chunk", str(CHUNK), "--out", OUT]
        if ALLOW_LONG:
            cmd.append("--allow-long")
        if OVERWRITE:
            cmd.append("--overwrite")
        if DENIAL_TEXT:
            cmd += ["--denial-text", DENIAL_TEXT]
        print("\n$", " ".join(cmd), flush=True)
        t0 = time.time()
        rc = stream(cmd)
        print(f"\n[{tag}] exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
        if rc != 0:
            print("fix the error above and re-run this cell; finished arms and chunks are kept")
        status(tag)
    show(tag)

In [ ]:
run_model("base")

In [ ]:
run_model("text_rl")

In [ ]:
run_model("tool_rl")

## 6. Analyze

`summary` is the headline table, one row per arm. `trunc_pct`, `no_boxed_pct`,
`code_pct` and `fake_out_pct` are the failure modes that would make a pass@k gap
uninterpretable, so read them before the curves. Under `prompt_off`, `calls_mean`
is how often the model wrote code after being told not to. `arith_err_pct` is the direct skill metric
(exact-rational check of every `a op b = c` in the prose, code blocks excluded);
`arith_coverage_pct` says how many samples contained anything to check.

In [ ]:
import sys, importlib
sys.path.insert(0, "src")
import analyze as an
importlib.reload(an)
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

df = load_run()   # current config's files only; strays from older runs are listed and skipped
print(len(df), "samples,", df.problem_id.nunique(), "problems,", df.arm.nunique(), "arms")
an.summary(df)

In [ ]:
an.passk_table(df).round(3)

In [ ]:
fig = an.plot_passk(df, title=f"{RUN_NAME} / {DATASET}: pass@k by condition")
fig.savefig(f"{OUT}/{DATASET}/passk.png", bbox_inches="tight")

In [ ]:
for cond in an.COND_ORDER:
    if cond in set(df.condition):
        an.verdict(df, cond)
        print()

## 7. Reading the result

- **Read within a column.** Each condition puts all three models under the same
  prompt and the same harness, so `verdict` compares them per condition.
  `env_denied` is the headline: every model asked for code, nobody gets to run it.
- **prompt_off tests compliance too.** `calls_mean` on those rows is how often a
  model wrote code after being told not to. A tool-RL model that cannot stop is
  itself a finding.
- **Compare to text_rl, not just base.** RL alone sharpens pass@1 and can lower
  high-k pass@k (the sharpening confound). The question is whether tool-RL sits
  below text-RL.
- **Check failure modes first.** If an arm has 30% truncation or never boxes an
  answer, its curve measures format collapse.
- **Arithmetic error rate** is the skill the interpreter replaces. Tool-RL above
  text-RL here, at the same accuracy, is atrophy of exactly that skill.

Then: flip `SMOKE=False`, raise `N_SAMPLES` to 32 or 64 for the high-k tail, add
`aime24`/`aime25` with `N_SAMPLES=64`, and move to `math-7b` (the CIR checkpoint
reproduces Fig. 6 with the control added). Per-subject breakdowns are one
`df.groupby(["tag","condition","subject"]).correct.mean()` away.

**Caveats.** `tool_on` executes model-written Python on this VM with a 10 s
timeout; fine for Colab, not for a shared machine. The CIR prompt's turn placement
is assumed. The ZeroTIR tool prompt is unverified (its repo has no card).
Qwen2.5-Math's native context is 4096, so long solutions truncate; the rate is in
the table.

In [ ]:
# Pull the trajectories to your laptop: every JSONL row carries the full generation
# text plus call / denial / truncation counts. Uncomment, run, then unzip locally and
# point analyze.load_results() at the <run>/<dataset> folder.
#
# from google.colab import files
# import shutil
# shutil.make_archive(f"/content/{RUN_NAME}", "zip", RESULTS, RUN_NAME)
# files.download(f"/content/{RUN_NAME}.zip")